# Diffraction Geometry And Kinematic Spots

PyTex currently treats diffraction through explicit geometry, reciprocal-space
primitives, and kinematic spot simulation. This notebook shows the geometry layer
and the current simulation surface.


In [ ]:
from pathlib import Path
import tempfile

import numpy as np

from pytex import (
    AcquisitionGeometry,
    AtomicSite,
    BenchmarkManifest,
    build_crystal_scene,
    CalibrationRecord,
    CrystalCellOverlay,
    CrystalDirection,
    CrystalDirectionOverlay,
    CrystalMap,
    CrystalPlane,
    CrystalPlaneOverlay,
    DirectionAnnotationStyle,
    DiffractionGeometry,
    EulerSet,
    ExperimentManifest,
    FrameDomain,
    FrameTransform,
    Handedness,
    InversePoleFigure,
    KernelSpec,
    KinematicSimulation,
    Lattice,
    get_phase_fixture,
    list_phase_fixtures,
    list_style_themes,
    MeasurementQuality,
    MillerIndex,
    ODF,
    Orientation,
    OrientationRelationship,
    OrientationSet,
    Phase,
    PhaseTransformationRecord,
    PoleFigure,
    PowderPattern,
    PowderReflection,
    ReferenceFrame,
    read_validation_manifest,
    read_workflow_result_manifest,
    resolve_style,
    RadiationSpec,
    Rotation,
    ScatteringSetup,
    SymmetrySpec,
    TransformationVariant,
    UnitCell,
    ValidationManifest,
    VectorSet,
    WorkflowResultManifest,
    ZoneAxis,
    PlaneAnnotationStyle,
    generate_saed_pattern,
    generate_xrd_pattern,
    normalize_ebsd,
    plot_odf,
    plot_crystal_structure_3d,
    plot_inverse_pole_figure,
    plot_ipf_map,
    plot_orientations,
    plot_kam_map,
    plot_pole_figure,
    plot_saed_pattern,
    plot_symmetry_elements,
    plot_symmetry_orbit,
    plot_vector_set,
    plot_xrd_pattern,
)


def make_crystal_frame():
    return ReferenceFrame(
        "crystal",
        FrameDomain.CRYSTAL,
        ("a", "b", "c"),
        Handedness.RIGHT,
    )


def make_context():
    crystal = make_crystal_frame()
    specimen = ReferenceFrame(
        "specimen",
        FrameDomain.SPECIMEN,
        ("x", "y", "z"),
        Handedness.RIGHT,
    )
    map_frame = ReferenceFrame(
        "map",
        FrameDomain.MAP,
        ("i", "j", "k"),
        Handedness.RIGHT,
    )
    detector = ReferenceFrame(
        "detector",
        FrameDomain.DETECTOR,
        ("u", "v", "n"),
        Handedness.RIGHT,
    )
    lab = ReferenceFrame(
        "lab",
        FrameDomain.LABORATORY,
        ("X", "Y", "Z"),
        Handedness.RIGHT,
    )
    phase = get_phase_fixture("ni_fcc").load_phase(crystal_frame=crystal)
    return crystal, specimen, map_frame, detector, lab, phase


def describe_phase_fixture(fixture_id):
    record = get_phase_fixture(fixture_id)
    return {
        "fixture_id": record.fixture_id,
        "display_name": record.display_name,
        "artifact_path": str(record.artifact_path),
        "metadata_path": str(record.metadata_path),
        "intended_uses": tuple(record.metadata["intended_uses"]),
    }


def load_zr_hcp_phase():
    return get_phase_fixture("zr_hcp").load_phase(crystal_frame=make_crystal_frame())


def load_diamond_phase():
    return get_phase_fixture("diamond").load_phase(crystal_frame=make_crystal_frame())


def publication_crystal_style():
    return {
        "crystal": {
            "atom_radius_scale": 0.5,
            "atom_edgewidth": 0.0,
            "atom_surface_resolution": 34,
            "bond_surface_resolution": 28,
            "bond_alpha": 0.72,
            "bond_color": "#7c8ea3",
            "atom_specular_strength": 0.42,
            "light_specular": 0.4,
        }
    }


In [ ]:
crystal, specimen, map_frame, detector, lab, phase = make_context()

geometry = DiffractionGeometry(
    detector_frame=detector,
    specimen_frame=specimen,
    laboratory_frame=lab,
    beam_energy_kev=200.0,
    camera_length_mm=150.0,
    pattern_center=np.array([0.5, 0.5, 0.7]),
    detector_pixel_size_um=(50.0, 50.0),
    detector_shape=(512, 512),
    scattering_setup=ScatteringSetup(laboratory_frame=lab, beam_energy_kev=200.0),
)

plane = CrystalPlane(miller=MillerIndex([1, 1, 1], phase=phase), phase=phase)
print("Ring radius (mm):", geometry.ring_radius_mm_for_plane(plane))


In [ ]:
orientation = Orientation(
    rotation=Rotation.identity(),
    crystal_frame=crystal,
    specimen_frame=specimen,
    symmetry=phase.symmetry,
    phase=phase,
)

simulation = KinematicSimulation.simulate_spots(
    geometry,
    phase,
    np.array(
        [
            [1, 0, 0],
            [1, 1, 0],
            [1, 1, 1],
            [2, 0, 0],
        ]
    ),
    orientation=orientation,
)

print("Accepted spots:", len(simulation.accepted_spots()))
print("Reflection families:", len(simulation.reflection_families))


In [ ]:
experiment = geometry.to_experiment_manifest(source_system="pytex", phase=phase)
print(experiment.to_dict()["modality"])
print(experiment.to_dict()["metadata"]["camera_length_mm"])
